# 先转数据格式，然后加载

In [ ]:
# from tahoe_x1.model import ComposerTX
# import scanpy as sc

# # Load model from Hugging Face in a single line
# # Options: "70m", "1b", or "3b"
# local_model_path = "/data/boom/Virtual_Organoid/Gene_encoder/Tahoe-x1/3b-model/"
# model, vocab, model_cfg, collator_cfg = ComposerTX.from_hf(
#     repo_id=local_model_path,
#     model_size="3b"
# )

# # Generate embeddings (see tutorials for full example)
# # Cell embeddings are stored in adata.obsm

/home/bob/anaconda3/envs/PhenoProfiler/lib/python3.11/site-packages/llmfoundry/callbacks/env_logging_callback.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
import pandas as pd
import scanpy as sc
import anndata
import os
import glob

# --- 1. 定义您的文件路径 ---

parquet_dir = "/data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/"
h5ad_dir = parquet_dir  # 保存在同一目录

# ！！！重要：确保这个路径是正确的！！！
gene_info_path = "/home/bob/boom/AIVC/VOBench/Gene_encoder/Bulk_Encoder/BulkFormer/data/bulkformer_gene_info.csv" 

print(f"输入目录 (Parquet): {parquet_dir}")
print(f"输出目录 (h5ad):   {h5ad_dir}\n")

# --- 2. 加载基因映射表 (一次性) ---
try:
    gene_info_df = pd.read_csv(gene_info_path)
    # 创建 符号 -> Ensembl ID 的映射字典
    symbol_to_ensg_map = gene_info_df.drop_duplicates(
        subset=['gene_symbol']
    ).set_index('gene_symbol')['ensg_id']
    print(f"已成功加载 {len(symbol_to_ensg_map)} 个基因的映射信息。")
except FileNotFoundError:
    print(f"错误: 找不到基因信息文件: {gene_info_path}")
    print("请确保 'data' 目录位于正确位置，并且该文件存在。")
    exit()

# --- 3. 查找所有 .parquet 文件 ---
parquet_files = glob.glob(os.path.join(parquet_dir, "*.parquet"))

if not parquet_files:
    print(f"在 {parquet_dir} 中未找到 .parquet 文件。")
    exit()

print(f"\n--- 发现了 {len(parquet_files)} 个 .parquet 文件。开始处理... ---")

# --- 4. 循环处理每个文件 ---
for parquet_path in parquet_files:
    try:
        base_name = os.path.basename(parquet_path)
        file_name_without_ext = os.path.splitext(base_name)[0]
        h5ad_path = os.path.join(h5ad_dir, f"{file_name_without_ext}.h5ad")
        
        # print(f"\n正在处理: {base_name}")
        
        # 加载 Parquet 文件
        data_df = pd.read_parquet(parquet_path)
        
        # 将 DataFrame 转换为 AnnData 对象
        adata = anndata.AnnData(data_df)
        
        # --- 关键步骤：添加 tahoe-x1 所需的元数据 ---
        
        # 1. (修复 gene_id_key)
        # 将基因符号 (adata.var_names) 映射到 'ensembl_id'
        adata.var['ensembl_id'] = adata.var_names.map(symbol_to_ensg_map)
        
        # 2. (修复 cell_type_key)
        # ！！！这是更新的逻辑！！！
        # 将 index (adata.obs_names) 中的细胞系名称复制到 'cell_type' 列
        adata.obs['cell_type'] = adata.obs_names
        
        # --- 元数据添加完毕 ---

        # 检查 (可选)
        # print(f"  adata.var head (基因元数据): \n{adata.var.head()}")
        # print(f"  adata.obs head (细胞元数据): \n{adata.obs.head()}")

        # 将 AnnData 对象保存为 .h5ad 文件 (覆盖旧文件)
        adata.write_h5ad(h5ad_path)
        
        print(f"  -> 已成功保存到: {h5ad_path} (已添加 'ensembl_id' 和 'cell_type' 列)")

    except Exception as e:
        print(f"  -> 处理 {base_name} 时出错: {e}")

print(f"\n--- 批量转换完成! ---")

输入目录 (Parquet): /data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/
输出目录 (h5ad):   /data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/

已成功加载 20007 个基因的映射信息。

--- 发现了 10 个 .parquet 文件。开始处理... ---

正在处理: GDSC.parquet


... storing 'ensembl_id' as categorical


  adata.var head (基因元数据): 
                 ensembl_id
feature_id                 
ZNF286A     ENSG00000187607
A1BG        ENSG00000121410
A1CF        ENSG00000148584
A2M         ENSG00000175899
A2ML1       ENSG00000166535
  adata.obs head (细胞元数据): 
       cell_type
CAL120    CAL120
DMS114    DMS114
CAL51      CAL51
H2869      H2869
H290        H290
  -> 已成功保存到: /data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/GDSC.h5ad (已添加 'ensembl_id' 和 'cell_type' 列)

正在处理: UMPDO3.parquet


... storing 'ensembl_id' as categorical


  adata.var head (基因元数据): 
                ensembl_id
feature_id                
DDX11L16               NaN
WASH7P                 NaN
MIR1302-2HG            NaN
ENSG00000241860        NaN
CICP27                 NaN
  adata.obs head (细胞元数据): 
                         cell_type
UMPDO-SAMPLE-304  UMPDO-SAMPLE-304
UMPDO-SAMPLE-261  UMPDO-SAMPLE-261
UMPDO-SAMPLE-260  UMPDO-SAMPLE-260
UMPDO-SAMPLE-211  UMPDO-SAMPLE-211
UMPDO-SAMPLE-210  UMPDO-SAMPLE-210
  -> 已成功保存到: /data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/UMPDO3.h5ad (已添加 'ensembl_id' 和 'cell_type' 列)

正在处理: UMPDO1.parquet


...

  adata.var head (基因元数据): 
                 ensembl_id
feature_id                 
TSPAN6      ENSG00000000003
TNMD        ENSG00000000005
DPM1        ENSG00000000419
SCYL3       ENSG00000000457
C1orf112    ENSG00000000460
  adata.obs head (细胞元数据): 
         cell_type
LC1SQSF    LC1SQSF
COGAR359  COGAR359
COLO794    COLO794
KKU213      KKU213
RT4            RT4
  -> 已成功保存到: /data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/CCLE.h5ad (已添加 'ensembl_id' 和 'cell_type' 列)

正在处理: Xeva.parquet


... storing 'ensembl_id' as categorical


  adata.var head (基因元数据): 
                 ensembl_id
feature_id                 
A1BG        ENSG00000121410
A1BG-AS1                NaN
A1CF        ENSG00000148584
A2LD1                   NaN
A2M         ENSG00000175899
  adata.obs head (细胞元数据): 
       cell_type
X-1004    X-1004
X-1008    X-1008
X-1027    X-1027
X-1119    X-1119
X-1156    X-1156
  -> 已成功保存到: /data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/Xeva.h5ad (已添加 'ensembl_id' 和 'cell_type' 列)

正在处理: NCI60.parquet


... storing 'ensembl_id' as categorical


  adata.var head (基因元数据): 
                 ensembl_id
feature_id                 
A1BG        ENSG00000121410
A1BG-AS1                NaN
A1CF        ENSG00000148584
A2M         ENSG00000175899
A2M-AS1                 NaN
  adata.obs head (细胞元数据): 
         cell_type
BT549        BT549
HS578T      HS578T
MCF7          MCF7
MDAMB231  MDAMB231
T47D          T47D
  -> 已成功保存到: /data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/NCI60.h5ad (已添加 'ensembl_id' 和 'cell_type' 列)

--- 批量转换完成! ---


In [4]:
import pandas as pd
import scanpy as sc
import anndata
import os
import glob

# Load your single-cell data
adata = sc.read_h5ad("/data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/UMPDO2.h5ad")
adata

AnnData object with n_obs × n_vars = 34 × 36447
    obs: 'cell_type'
    var: 'ensembl_id'

In [1]:
from omegaconf import OmegaConf as om
from scripts.inference.predict_embeddings import predict_embeddings

cfg = {
    "model_name": "Tx1-70m",
    "paths": {
        "hf_repo_id": "tahoebio/Tahoe-x1",
        "hf_model_size": "70m",
        "adata_input": "/data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/UMPDO2.h5ad",
    },
    "data": {
        "cell_type_key": "cell_type",
        "gene_id_key": "ensembl_id"
    },
    "predict": {
        "seq_len_dataset": 2048,
        "return_gene_embeddings": False,
    }
}

cfg = om.create(cfg)
adata = predict_embeddings(cfg)

# Access embeddings
cell_embeddings = adata.obsm["Tx1-70m"]

/home/bob/anaconda3/envs/PhenoProfiler/lib/python3.11/site-packages/llmfoundry/callbacks/env_logging_callback.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-11-15 13:27:14,881: [171334][MainThread]: INFO: scripts.inference.predict_embeddings: Loading model from Hugging Face repo tahoebio/Tahoe-x1, size 70m
2025-11-15 13:27:15,089: [171334][MainThread]: INFO: tahoe_x1.model.model: MosaicML recommends using config.init_device="meta" with Composer + FSDP for faster initialization.
2025-11-15 13:27:15,887: [171334][MainThread]: INFO: scripts.inference.predict_embeddings: Loading AnnData file…
/home/bob/boom/AIVC/VOBench/Gene_encoder/tahoe-x1/scripts/inference/predict_embeddings.py:80: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initiali

Model is loaded with 12 transformer layers.
match 17133/36447 genes in vocabulary of size 62720.


2025-11-15 13:27:17,018: [171334][MainThread]: INFO: composer.trainer.trainer: Stepping schedulers every batch. To step schedulers every epoch, set `step_schedulers_every_batch=False`.
2025-11-15 13:27:17,018: [171334][MainThread]: INFO: composer.trainer.trainer: Setting seed to 3626266692
2025-11-15 13:27:17,019: [171334][MainThread]: INFO: composer.utils.reproducibility: Setting seed to 3626266692
******************************
Config:
composer_commit_hash: None
composer_version: 0.32.1
node_name: unknown because NODENAME environment variable not set
num_gpus_per_node: 1
num_nodes: 1
rank_zero_seed: 3626266692

******************************
2025-11-15 13:27:17,563: [171334][MainThread]: INFO: scripts.inference.predict_embeddings: Aggregating embeddings…
2025-11-15 13:27:17,564: [171334][MainThread]: INFO: scripts.inference.predict_embeddings: Saving outputs…
2025-11-15 13:27:17,564: [171334][MainThread]: INFO: scripts.inference.predict_embeddings: Storing cell embeddings in adata.ob

In [2]:
cell_embeddings.shape

(34, 512)

# 循环处理 全部

In [1]:
import pandas as pd
import os
import glob
from omegaconf import OmegaConf as om

# --- 1. 导入 Tahoe-x1 函数 ---
# 
# ！！！重要！！！
# 这假设 'scripts' 目录在您的 Python 路径中，
# 或者您是从 'tahoe-x1' 的根目录运行此脚本的。
try:
    from scripts.inference.predict_embeddings import predict_embeddings
except ImportError:
    print("错误: 无法 'from scripts.inference.predict_embeddings import predict_embeddings'")
    print("请确保您是从 'tahoe-x1' 代码库的根目录运行此脚本，")
    print("或者 'scripts' 目录已添加到您的 PYTHONPATH。")
    exit()

# --- 2. 定义路径和基础配置 ---

# 输入目录 (包含 .h5ad 文件)
input_dir = "/data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/"

# 输出目录 (保存 .parquet 嵌入)
output_dir = "/data/boom/Virtual_Organoid/Gene_encoder/Tahoe_x1/"

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)

# 基础配置 (模板)
# 我们将在循环中动态设置 'paths.adata_input'
cfg_base = {
    "model_name": "Tx1-70m",  # 您正在使用的模型名 (用于 .obsm 密钥)
    "paths": {
        "hf_repo_id": "tahoebio/Tahoe-x1",
        "hf_model_size": "70m",  # 确保这与 model_name 匹配
        "adata_input": "",  # <--- 将在循环中填充
    },
    "data": {
        "cell_type_key": "cell_type",
        "gene_id_key": "ensembl_id"
    },
    "predict": {
        "seq_len_dataset": 2048,
        "return_gene_embeddings": False,
    }
}

print(f"输入目录: {input_dir}")
print(f"输出目录: {output_dir}")

# --- 3. 查找所有 .h5ad 文件 ---
h5ad_files = glob.glob(os.path.join(input_dir, "*.h5ad"))

if not h5ad_files:
    print(f"在 {input_dir} 中未找到 .h5ad 文件。")
    exit()

print(f"\n--- 发现了 {len(h5ad_files)} 个 .h5ad 文件。开始处理... ---")

# --- 4. 循环处理每个文件 ---
for h5ad_path in h5ad_files:
    try:
        # 获取不带扩展名的文件名 (例如: "UMPDO2")
        base_name = os.path.basename(h5ad_path)
        name = os.path.splitext(base_name)[0]
        
        print(f"\n正在处理: {base_name}")

        # --- a. 动态创建此文件的配置 ---
        current_cfg_dict = cfg_base.copy()
        current_cfg_dict["paths"]["adata_input"] = h5ad_path
        
        cfg = om.create(current_cfg_dict)

        # --- b. 运行嵌入生成 ---
        # print("  -> 正在调用 predict_embeddings...")
        adata = predict_embeddings(cfg)

        # --- c. 提取嵌入 ---
        # 使用 cfg 中的 'model_name' 作为 .obsm 中的密钥
        model_key = cfg.model_name
        if model_key not in adata.obsm:
            print(f"  -> 错误: 在 adata.obsm 中未找到密钥 '{model_key}'")
            continue # 跳到下一个文件
            
        cell_embeddings = adata.obsm[model_key]

        # --- d. 转换为 DataFrame ---
        res1_df = pd.DataFrame(
            cell_embeddings,
            index=adata.obs_names  # 使用细胞系名称作为索引
        )

        # --- e. 定义输出路径并保存 ---
        # 按照您的要求: {name}_embeddings.parquet
        output_path = os.path.join(output_dir, f"{name}_embeddings.parquet")
        
        res1_df.to_parquet(output_path)
        
        print(f"  -> 成功! 已保存嵌入到: {output_path}")

    except Exception as e:
        print(f"  -> 处理 {base_name} 时出错: {e}")

print(f"\n--- 批量处理完成! ---")

/home/bob/anaconda3/envs/PhenoProfiler/lib/python3.11/site-packages/llmfoundry/callbacks/env_logging_callback.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-11-15 13:34:54,995: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Loading model from Hugging Face repo tahoebio/Tahoe-x1, size 70m
2025-11-15 13:34:55,165: [171589][MainThread]: INFO: tahoe_x1.model.model: MosaicML recommends using config.init_device="meta" with Composer + FSDP for faster initialization.


输入目录: /data/boom/Virtual_Organoid/CLIO_processed/omics_mrna_raw/
输出目录: /data/boom/Virtual_Organoid/Gene_encoder/Tahoe_x1/

--- 发现了 10 个 .h5ad 文件。开始处理... ---

正在处理: Xeva.h5ad


2025-11-15 13:34:55,851: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Loading AnnData file…
/home/bob/boom/AIVC/VOBench/Gene_encoder/tahoe-x1/scripts/inference/predict_embeddings.py:80: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["id_in_vocab"] = [
2025-11-15 13:34:55,890: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Creating data loader…
2025-11-15 13:34:55,948: [171589][MainThread]: INFO: composer.utils.reproducibility: Setting seed to 2592087415
/home/bob/anaconda3/envs/PhenoProfiler/lib/python3.11/site-packages/composer/trainer/trainer.py:1224: UserWarning: No optimizer was specified. Defaulting to DecoupledSGDW(lr=0.1)
  warnings.warn((
2025-11-15 13:34:55,951: [171589][MainThread]: INFO: composer.trainer.trainer: Run name: 1763184895-stimulating-pig


Model is loaded with 12 transformer layers.
match 15835/19711 genes in vocabulary of size 62720.


2025-11-15 13:34:57,026: [171589][MainThread]: INFO: composer.trainer.trainer: Stepping schedulers every batch. To step schedulers every epoch, set `step_schedulers_every_batch=False`.
2025-11-15 13:34:57,026: [171589][MainThread]: INFO: composer.trainer.trainer: Setting seed to 2592087415
2025-11-15 13:34:57,026: [171589][MainThread]: INFO: composer.utils.reproducibility: Setting seed to 2592087415
******************************
Config:
composer_commit_hash: None
composer_version: 0.32.1
node_name: unknown because NODENAME environment variable not set
num_gpus_per_node: 1
num_nodes: 1
rank_zero_seed: 2592087415

******************************
2025-11-15 13:34:58,097: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Aggregating embeddings…
2025-11-15 13:34:58,098: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Saving outputs…
2025-11-15 13:34:58,098: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Storing cell embeddings in adata.ob

...

2025-11-15 13:35:16,733: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Loading AnnData file…
/home/bob/boom/AIVC/VOBench/Gene_encoder/tahoe-x1/scripts/inference/predict_embeddings.py:80: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["id_in_vocab"] = [
2025-11-15 13:35:16,759: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Creating data loader…
2025-11-15 13:35:16,777: [171589][MainThread]: INFO: composer.utils.reproducibility: Setting seed to 775024076
/home/bob/anaconda3/envs/PhenoProfiler/lib/python3.11/site-packages/composer/trainer/trainer.py:1224: UserWarning: No optimizer was specified. Defaulting to DecoupledSGDW(lr=0.1)
  warnings.warn((
2025-11-15 13:35:16,778: [171589][MainThread]: INFO: composer.trainer.trainer: Run name: 1763184916-lush-boa
2025-11-15 13:35:16,784: [171589][MainThread]: INFO: composer.trainer.trainer: Stepping schedulers every batch. To step schedulers 

Model is loaded with 12 transformer layers.
match 15337/24027 genes in vocabulary of size 62720.


2025-11-15 13:35:17,268: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Aggregating embeddings…
2025-11-15 13:35:17,268: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Saving outputs…
2025-11-15 13:35:17,269: [171589][MainThread]: INFO: scripts.inference.predict_embeddings: Storing cell embeddings in adata.obsm['Tx1-70m']


  -> 成功! 已保存嵌入到: /data/boom/Virtual_Organoid/Gene_encoder/Tahoe_x1/Tavor_embeddings.parquet

--- 批量处理完成! ---


/home/bob/boom/AIVC/VOBench/Gene_encoder/tahoe-x1/scripts/inference/predict_embeddings.py:172: ImplicitModificationWarning: Setting element `.obsm['Tx1-70m']` of view, initializing view as actual.
  adata.obsm[model_name] = cell_array


In [2]:
res1_df

,0,1,2,3,4,5,6,7,8,9,...,502,503,504,505,506,507,508,509,510,511
140630,-0.001308,0.005502,-0.061557,-0.017544,0.042096,-0.030026,0.000199,0.005021,0.024353,0.079069,...,0.003383,-0.014460,-0.000728,-0.053532,0.010026,0.042256,-0.002058,-0.129172,-0.002036,0.008221
140690,0.002976,0.010265,-0.025500,-0.006859,0.045538,-0.037911,0.000214,0.016770,0.017717,0.090524,...,-0.008006,-0.017106,0.001074,-0.041212,0.028361,0.010067,-0.002183,-0.143909,-0.000105,0.008414
140692,-0.001011,0.005244,-0.060944,-0.005897,0.039979,-0.037568,0.000591,0.000692,0.025329,0.075627,...,0.008373,-0.014279,-0.000463,-0.043095,0.037437,0.043959,-0.000886,-0.113159,-0.001641,0.008148
140699,0.000060,0.016643,-0.046087,-0.007365,0.048743,-0.028559,0.000341,-0.013733,0.027418,0.051023,...,-0.008637,-0.016904,0.000554,-0.048774,0.029108,-0.017073,-0.002148,-0.153292,-0.000606,0.008729
140700,-0.002571,0.009474,-0.059364,0.000120,0.041480,-0.039498,0.000919,0.007490,0.043488,0.095775,...,0.007922,-0.013755,-0.000277,-0.045249,0.050055,0.061296,-0.000020,-0.114255,-0.002004,0.009341
140743,0.000002,0.010947,-0.063441,-0.020574,0.039435,-0.025726,0.000833,0.001082,0.019042,0.076732,...,0.012867,-0.014501,0.000007,-0.047275,0.010425,0.023295,-0.002493,-0.124138,-0.001938,0.007326
140786,0.000976,0.013435,-0.041959,-0.017802,0.048054,-0.024277,0.000067,-0.024207,-0.019388,0.019687,...,0.010717,-0.015716,-0.001285,-0.003522,-0.037333,-0.023482,-0.002452,-0.123239,-0.002972,0.008094
140877,-0.001772,0.032000,-0.038686,-0.015148,0.050322,-0.023371,0.000205,0.001219,0.020327,0.050010,...,0.004752,-0.017018,-0.000146,-0.047150,0.029449,-0.000236,-0.003018,-0.158352,0.000419,0.009962
140884,-0.002273,-0.001052,-0.058638,-0.013006,0.037739,-0.025628,0.000750,0.006412,0.028473,0.070908,...,0.016179,-0.014498,-0.000847,-0.061151,0.019970,0.028300,-0.002603,-0.134877,-0.001749,0.008232
140899,0.001926,0.021669,-0.029908,-0.007681,0.049591,-0.029722,-0.001323,-0.020883,-0.022151,0.046632,...,-0.001405,-0.017322,0.001348,-0.026226,0.000800,-0.054584,-0.002415,-0.174313,-0.000110,0.007612
